# Notebook 21 — Agent Orchestration Patterns
## Routing and Control Flow

Beyond simple pipelines: **conditional routing**, **parallel fan-out/fan-in**, and **DAG-based execution**.

| Pattern | Description | Use Case |
|---------|-------------|----------|
| Router | Classify → dispatch to specialist | Multi-domain tasks |
| Conditional | Decision tree branching | Task-type routing |
| Fan-out/Fan-in | Parallel execution → aggregation | Independent subtasks |
| DAG | Directed graph of agents | Complex workflows |

**Prerequisites:** Notebook 01–20 foundations.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Beyond Pipelines — Why We Need Flexible Orchestration

Linear pipelines (A → B → C) break down when:
- Different inputs need **different processing paths**
- Tasks can be done **in parallel** for speed
- Complex workflows have **dependencies** (DAGs, not chains)

Real-world agent systems need **routing**, **branching**, and **graph execution**.

In [ ]:
# Base agent class for all orchestration patterns
class BaseAgent:
    """Minimal agent with name, role, and generate capability."""
    def __init__(self, name: str, role: str, specialty: str = "general"):
        self.name = name
        self.role = role
        self.specialty = specialty
        self.call_count = 0

    def run(self, task: str) -> Dict[str, Any]:
        self.call_count += 1
        messages = [
            {"role": "system", "content": f"You are {self.name}, a {self.role}. Specialty: {self.specialty}. Be concise (2-3 sentences)."},
            {"role": "user", "content": task}
        ]
        t0 = time.time()
        response = generate(messages, max_new_tokens=200)
        elapsed = time.time() - t0
        return {
            "agent": self.name,
            "specialty": self.specialty,
            "response": response,
            "time": round(elapsed, 2)
        }

    def __repr__(self):
        return f"Agent({self.name}, specialty={self.specialty})"

# Create specialist agents
code_agent = BaseAgent("CodeBot", "expert Python programmer", "code")
math_agent = BaseAgent("MathBot", "mathematics expert", "math")
writing_agent = BaseAgent("WriteBot", "professional writer", "writing")
research_agent = BaseAgent("ResearchBot", "research analyst", "research")

print("✓ Created 4 specialist agents:")
for a in [code_agent, math_agent, writing_agent, research_agent]:
    print(f"  {a}")

## 2. Router Agent — Classify and Dispatch

The router examines an incoming task, **classifies its type**, then dispatches to the appropriate specialist.

```
         ┌──────────┐
Task ──▶ │  Router  │──▶ CodeBot / MathBot / WriteBot / ResearchBot
         └──────────┘
              │
         (classifies)
```

In [ ]:
class RouterAgent:
    """Classifies incoming tasks and routes to the appropriate specialist."""

    def __init__(self, agents: List[BaseAgent]):
        self.agents = {a.specialty: a for a in agents}
        self.routing_log = []

    def classify(self, task: str) -> str:
        """Use LLM to classify task type."""
        specialties = list(self.agents.keys())
        messages = [
            {"role": "system", "content": f"""You are a task classifier. Classify the task into exactly one category.
Available categories: {specialties}
Reply with ONLY the category name, nothing else."""},
            {"role": "user", "content": task}
        ]
        result = generate(messages, max_new_tokens=20, temperature=0.1, do_sample=False)
        # Extract matching category
        result_lower = result.strip().lower()
        for s in specialties:
            if s in result_lower:
                return s
        return specialties[0]  # fallback

    def route(self, task: str) -> Dict[str, Any]:
        """Classify and route task to appropriate agent."""
        t0 = time.time()
        category = self.classify(task)
        classify_time = time.time() - t0

        agent = self.agents[category]
        result = agent.run(task)
        result["routed_to"] = category
        result["classify_time"] = round(classify_time, 2)

        self.routing_log.append({
            "task": task[:60],
            "category": category,
            "agent": agent.name
        })
        return result

router = RouterAgent([code_agent, math_agent, writing_agent, research_agent])
print(f"✓ RouterAgent created with {len(router.agents)} specialists: {list(router.agents.keys())}")

In [ ]:
# Test the router with different task types
test_tasks = [
    "Write a Python function that calculates the Fibonacci sequence",
    "What is the integral of x^2 * sin(x)?",
    "Write a compelling product description for a smart water bottle",
    "Summarize the key findings about transformer architectures in NLP",
]

print("=" * 70)
print("ROUTER AGENT — Task Classification & Dispatch")
print("=" * 70)

for task in test_tasks:
    result = router.route(task)
    print(f"\nTask: {task[:55]}...")
    print(f"  → Routed to: {result['routed_to']} ({result['agent']})")
    print(f"  → Classify: {result['classify_time']}s | Generate: {result['time']}s")
    print(f"  → Response: {result['response'][:120]}...")

## 3. Conditional Routing — Decision Tree Dispatch

Instead of LLM-based classification, use **rule-based** routing for speed and predictability.

```
Task ──▶ [keyword check]
           ├── code keywords? ──▶ CodeBot
           ├── math keywords? ──▶ MathBot
           ├── write keywords? ──▶ WriteBot
           └── default ──────────▶ ResearchBot
```

In [ ]:
class ConditionalRouter:
    """Rule-based routing using keyword patterns and decision trees."""

    def __init__(self, agents: Dict[str, BaseAgent]):
        self.agents = agents
        self.rules: List[Tuple[str, Callable[[str], bool]]] = []
        self.default_category = "research"
        self.route_log = []

    def add_rule(self, category: str, condition: Callable[[str], bool]):
        """Add a routing rule: if condition(task) is True, route to category."""
        self.rules.append((category, condition))

    def classify(self, task: str) -> str:
        task_lower = task.lower()
        for category, condition in self.rules:
            if condition(task_lower):
                return category
        return self.default_category

    def route(self, task: str) -> Dict[str, Any]:
        category = self.classify(task)
        agent = self.agents[category]
        result = agent.run(task)
        result["routed_to"] = category
        result["routing_method"] = "conditional"
        self.route_log.append({"task": task[:50], "category": category})
        return result

# Build conditional router with keyword rules
cond_router = ConditionalRouter({
    "code": code_agent, "math": math_agent,
    "writing": writing_agent, "research": research_agent
})

cond_router.add_rule("code", lambda t: any(kw in t for kw in [
    "python", "function", "code", "program", "algorithm", "implement", "debug", "class"
]))
cond_router.add_rule("math", lambda t: any(kw in t for kw in [
    "integral", "derivative", "equation", "calculate", "probability", "matrix", "proof"
]))
cond_router.add_rule("writing", lambda t: any(kw in t for kw in [
    "write", "essay", "story", "poem", "description", "blog", "article", "letter"
]))

print("✓ ConditionalRouter with 3 rules + default fallback")

In [ ]:
# Compare LLM router vs conditional router
print("=" * 70)
print("LLM ROUTER vs CONDITIONAL ROUTER — Speed Comparison")
print("=" * 70)

comparison_tasks = [
    "Write a Python class for a binary search tree",
    "Calculate the eigenvalues of a 3x3 matrix",
    "Compose a haiku about artificial intelligence",
    "What are the main theories about dark matter?",
]

for task in comparison_tasks:
    # LLM routing
    t0 = time.time()
    llm_cat = router.classify(task)
    llm_time = time.time() - t0

    # Conditional routing
    t0 = time.time()
    cond_cat = cond_router.classify(task)
    cond_time = time.time() - t0

    print(f"\nTask: {task[:55]}")
    print(f"  LLM router  → {llm_cat:10s} ({llm_time:.3f}s)")
    print(f"  Cond router → {cond_cat:10s} ({cond_time:.6f}s)")
    print(f"  Speedup: {llm_time / max(cond_time, 1e-6):.0f}x")

## 4. Parallel Fan-Out / Fan-In

**Fan-out**: Send a task to multiple agents simultaneously.
**Fan-in**: Collect and aggregate all results.

```
         ┌── Agent A ──┐
Task ──▶─┤── Agent B ──├──▶ Aggregator ──▶ Final
         └── Agent C ──┘
```

Useful when you want **multiple perspectives** on the same task.

In [ ]:
class FanOutFanIn:
    """Execute task across multiple agents, then aggregate results."""

    def __init__(self, agents: List[BaseAgent], aggregator: Optional[BaseAgent] = None):
        self.agents = agents
        self.aggregator = aggregator

    def fan_out(self, task: str) -> List[Dict[str, Any]]:
        """Send task to all agents (simulated parallel — sequential in Colab)."""
        results = []
        print(f"Fan-out: dispatching to {len(self.agents)} agents...")
        for agent in self.agents:
            result = agent.run(task)
            results.append(result)
            print(f"  ✓ {agent.name} responded ({result['time']}s)")
        return results

    def fan_in(self, task: str, results: List[Dict[str, Any]]) -> str:
        """Aggregate results from multiple agents."""
        if self.aggregator is None:
            return self._simple_aggregate(results)

        # Use aggregator agent to synthesize
        combined = "\n".join([
            f"[{r['agent']} ({r['specialty']})]: {r['response']}"
            for r in results
        ])
        agg_task = f"""Original task: {task}

Multiple specialists provided these responses:
{combined}

Synthesize these into a single coherent response. Keep the best insights from each."""
        agg_result = self.aggregator.run(agg_task)
        return agg_result["response"]

    def _simple_aggregate(self, results: List[Dict[str, Any]]) -> str:
        return "\n---\n".join([
            f"**{r['agent']}**: {r['response']}" for r in results
        ])

    def execute(self, task: str) -> Dict[str, Any]:
        t0 = time.time()
        results = self.fan_out(task)
        synthesis = self.fan_in(task, results)
        total_time = time.time() - t0
        return {
            "individual_results": results,
            "synthesis": synthesis,
            "total_time": round(total_time, 2),
            "num_agents": len(self.agents)
        }

# Create aggregator agent
aggregator = BaseAgent("Synthesizer", "expert at combining multiple viewpoints", "synthesis")

fan_system = FanOutFanIn(
    agents=[code_agent, research_agent, writing_agent],
    aggregator=aggregator
)
print("✓ FanOutFanIn system: 3 specialists + 1 aggregator")

In [ ]:
# Execute fan-out/fan-in
task = "Explain how to build a recommendation system for an e-commerce platform"

print("=" * 70)
print("FAN-OUT / FAN-IN EXECUTION")
print("=" * 70)
print(f"Task: {task}\n")

result = fan_system.execute(task)

print(f"\n{'=' * 70}")
print("INDIVIDUAL RESPONSES:")
print("=" * 70)
for r in result["individual_results"]:
    print(f"\n[{r['agent']} — {r['specialty']}]")
    print(f"  {r['response'][:200]}")

print(f"\n{'=' * 70}")
print("SYNTHESIZED RESPONSE:")
print("=" * 70)
print(result["synthesis"][:500])
print(f"\nTotal time: {result['total_time']}s across {result['num_agents']} agents + aggregator")

## 5. DAG-Based Execution — Directed Acyclic Graph of Agents

Model complex workflows as a **directed acyclic graph** (DAG):
- **Nodes** = agents/tasks
- **Edges** = data dependencies
- Execute in **topological order** (respecting dependencies)

```
  Research ──▶ Analyze ──▶ Synthesize
      │                      ▲
      └───▶ Code Review ─────┘
```

In [ ]:
class DAGNode:
    """A node in the execution DAG."""
    def __init__(self, node_id: str, agent: BaseAgent, task_template: str):
        self.node_id = node_id
        self.agent = agent
        self.task_template = task_template  # can reference {prev_node_id}
        self.result: Optional[Dict[str, Any]] = None

    def __repr__(self):
        status = "✓" if self.result else "○"
        return f"[{status}] {self.node_id} ({self.agent.name})"


class DAGExecutor:
    """Execute a DAG of agents respecting dependencies."""

    def __init__(self):
        self.nodes: Dict[str, DAGNode] = {}
        self.edges: List[Tuple[str, str]] = []  # (from, to)

    def add_node(self, node_id: str, agent: BaseAgent, task_template: str):
        self.nodes[node_id] = DAGNode(node_id, agent, task_template)

    def add_edge(self, from_id: str, to_id: str):
        """Add dependency: to_id depends on from_id."""
        self.edges.append((from_id, to_id))

    def _topological_sort(self) -> List[str]:
        """Kahn's algorithm for topological sorting."""
        in_degree = defaultdict(int)
        adjacency = defaultdict(list)

        for node_id in self.nodes:
            in_degree[node_id] = 0

        for src, dst in self.edges:
            adjacency[src].append(dst)
            in_degree[dst] += 1

        queue = [n for n in self.nodes if in_degree[n] == 0]
        order = []

        while queue:
            node = queue.pop(0)
            order.append(node)
            for neighbor in adjacency[node]:
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)

        if len(order) != len(self.nodes):
            raise ValueError("DAG has a cycle!")
        return order

    def _get_predecessors(self, node_id: str) -> List[str]:
        return [src for src, dst in self.edges if dst == node_id]

    def execute(self, initial_context: str = "") -> Dict[str, Any]:
        """Execute DAG in topological order."""
        order = self._topological_sort()
        print(f"Execution order: {' → '.join(order)}")
        print("-" * 50)

        results = {}
        t0 = time.time()

        for node_id in order:
            node = self.nodes[node_id]

            # Build task by substituting predecessor results
            task = node.task_template
            if "{context}" in task:
                task = task.replace("{context}", initial_context)
            for pred_id in self._get_predecessors(node_id):
                pred_result = results[pred_id]["response"]
                task = task.replace(f"{{{pred_id}}}", pred_result)

            print(f"Executing: {node_id} ({node.agent.name})...")
            result = node.agent.run(task)
            results[node_id] = result
            node.result = result
            print(f"  ✓ Done ({result['time']}s)")

        return {
            "execution_order": order,
            "results": results,
            "total_time": round(time.time() - t0, 2)
        }

print("✓ DAGExecutor class defined")

In [ ]:
# Build and execute a DAG workflow
dag = DAGExecutor()

# Add nodes
dag.add_node("research", research_agent,
    "Research the topic of building AI chatbots. Provide key findings. Context: {context}")
dag.add_node("code_review", code_agent,
    "Based on this research, suggest a code architecture for an AI chatbot: {research}")
dag.add_node("analysis", math_agent,
    "Analyze the computational requirements for this chatbot architecture: {research}")
dag.add_node("synthesis", writing_agent,
    "Write a summary combining these inputs:\nArchitecture: {code_review}\nAnalysis: {analysis}")

# Add edges (dependencies)
dag.add_edge("research", "code_review")
dag.add_edge("research", "analysis")
dag.add_edge("code_review", "synthesis")
dag.add_edge("analysis", "synthesis")

print("DAG Structure:")
print("  research ──▶ code_review ──▶ synthesis")
print("      │                         ▲")
print("      └──────▶ analysis ────────┘")
print()

dag_result = dag.execute("Building a production AI chatbot")
print(f"\nTotal DAG execution: {dag_result['total_time']}s")

## 6. Lightweight Graph Executor — Reusable Pattern

A more general graph executor that supports **named connections** and **data transformation** between nodes.

In [ ]:
class GraphExecutor:
    """Lightweight graph executor with named connections and transforms."""

    def __init__(self, name: str = "workflow"):
        self.name = name
        self.nodes: Dict[str, Callable] = {}
        self.connections: List[Dict[str, Any]] = []
        self.results: Dict[str, Any] = {}

    def add_node(self, name: str, fn: Callable):
        """Add a processing node (any callable)."""
        self.nodes[name] = fn

    def connect(self, source: str, target: str, transform: Optional[Callable] = None):
        """Connect source output to target input, with optional transform."""
        self.connections.append({
            "source": source, "target": target,
            "transform": transform or (lambda x: x)
        })

    def _get_inputs(self, node_name: str) -> Dict[str, Any]:
        inputs = {}
        for conn in self.connections:
            if conn["target"] == node_name and conn["source"] in self.results:
                source_data = self.results[conn["source"]]
                inputs[conn["source"]] = conn["transform"](source_data)
        return inputs

    def _topo_sort(self) -> List[str]:
        in_deg = {n: 0 for n in self.nodes}
        adj = defaultdict(list)
        for c in self.connections:
            adj[c["source"]].append(c["target"])
            in_deg[c["target"]] += 1
        queue = [n for n in self.nodes if in_deg[n] == 0]
        order = []
        while queue:
            n = queue.pop(0)
            order.append(n)
            for nb in adj[n]:
                in_deg[nb] -= 1
                if in_deg[nb] == 0:
                    queue.append(nb)
        return order

    def execute(self, initial_input: Any = None) -> Dict[str, Any]:
        order = self._topo_sort()
        print(f"[{self.name}] Executing: {' → '.join(order)}")

        for node_name in order:
            inputs = self._get_inputs(node_name)
            if not inputs and initial_input is not None:
                inputs = {"_initial": initial_input}

            fn = self.nodes[node_name]
            self.results[node_name] = fn(inputs)
            print(f"  ✓ {node_name} complete")

        return self.results

# Helper: wrap an agent as a graph node function
def agent_node(agent: BaseAgent, task_builder: Callable):
    """Wrap an agent into a graph node function."""
    def node_fn(inputs: Dict) -> str:
        task = task_builder(inputs)
        result = agent.run(task)
        return result["response"]
    return node_fn

print("✓ GraphExecutor defined")

In [ ]:
# Build a workflow with GraphExecutor
workflow = GraphExecutor("chatbot_design")

workflow.add_node("gather_requirements",
    agent_node(research_agent, lambda inp: f"List 5 key requirements for building a customer support chatbot. {inp.get('_initial', '')}"))

workflow.add_node("design_architecture",
    agent_node(code_agent, lambda inp: f"Design a system architecture for a chatbot with these requirements: {inp.get('gather_requirements', 'N/A')[:300]}"))

workflow.add_node("estimate_costs",
    agent_node(math_agent, lambda inp: f"Estimate compute costs for this chatbot architecture: {inp.get('gather_requirements', 'N/A')[:300]}"))

workflow.add_node("final_report",
    agent_node(writing_agent, lambda inp: f"Write a brief project proposal combining: Architecture: {inp.get('design_architecture', 'N/A')[:200]} Costs: {inp.get('estimate_costs', 'N/A')[:200]}"))

workflow.connect("gather_requirements", "design_architecture")
workflow.connect("gather_requirements", "estimate_costs")
workflow.connect("design_architecture", "final_report")
workflow.connect("estimate_costs", "final_report")

print("Workflow graph:")
print("  requirements ──▶ architecture ──▶ final_report")
print("       │                             ▲")
print("       └──────▶ costs ───────────────┘\n")

results = workflow.execute("E-commerce customer support")
print(f"\nFinal report preview:\n{results['final_report'][:400]}")

## 7. Complex Orchestration — Multi-Step with Branching

A more complex example: **task triage** → **parallel specialist work** → **quality check** → **merge or retry**.

In [ ]:
class OrchestrationEngine:
    """Complex orchestration with triage, parallel dispatch, and quality check."""

    def __init__(self, specialists: Dict[str, BaseAgent], reviewer: BaseAgent):
        self.specialists = specialists
        self.reviewer = reviewer
        self.execution_trace = []

    def triage(self, task: str) -> List[str]:
        """Determine which specialists are needed."""
        messages = [
            {"role": "system", "content": f"""Analyze the task and determine which specialists are needed.
Available: {list(self.specialists.keys())}
Reply with a comma-separated list of needed specialists. Only include those truly relevant."""},
            {"role": "user", "content": task}
        ]
        result = generate(messages, max_new_tokens=50, temperature=0.1, do_sample=False)
        needed = []
        for spec in self.specialists:
            if spec.lower() in result.lower():
                needed.append(spec)
        return needed if needed else [list(self.specialists.keys())[0]]

    def quality_check(self, task: str, responses: Dict[str, str]) -> Dict[str, Any]:
        """Review quality of responses."""
        combined = "\n".join([f"[{k}]: {v[:200]}" for k, v in responses.items()])
        review_task = f"""Review these specialist responses for quality.
Task: {task}
Responses:
{combined}

Rate overall quality 1-10 and suggest if we need revisions. Reply as: SCORE: X/10 followed by brief assessment."""
        result = self.reviewer.run(review_task)
        # Parse score
        score_match = re.search(r'(\d+)/10', result["response"])
        score = int(score_match.group(1)) if score_match else 5
        return {"score": score, "review": result["response"]}

    def execute(self, task: str) -> Dict[str, Any]:
        self.execution_trace = []
        t0 = time.time()

        # Step 1: Triage
        needed = self.triage(task)
        self.execution_trace.append(f"Triage → {needed}")
        print(f"Step 1 - Triage: {needed}")

        # Step 2: Parallel specialist work
        responses = {}
        for spec_name in needed:
            agent = self.specialists[spec_name]
            result = agent.run(task)
            responses[spec_name] = result["response"]
            self.execution_trace.append(f"{spec_name} → responded")
            print(f"Step 2 - {spec_name}: responded ({result['time']}s)")

        # Step 3: Quality check
        review = self.quality_check(task, responses)
        self.execution_trace.append(f"Review → {review['score']}/10")
        print(f"Step 3 - Quality: {review['score']}/10")

        return {
            "specialists_used": needed,
            "responses": responses,
            "quality_score": review["score"],
            "review": review["review"],
            "total_time": round(time.time() - t0, 2),
            "trace": self.execution_trace
        }

engine = OrchestrationEngine(
    specialists={"code": code_agent, "math": math_agent, "writing": writing_agent, "research": research_agent},
    reviewer=BaseAgent("Reviewer", "quality reviewer who evaluates completeness and accuracy", "review")
)

print("✓ OrchestrationEngine with 4 specialists + 1 reviewer")

In [ ]:
# Run complex orchestration
task = "Design a machine learning pipeline that predicts customer churn, including data preprocessing, model selection, and a report for stakeholders"

print("=" * 70)
print("COMPLEX ORCHESTRATION")
print("=" * 70)
print(f"Task: {task}\n")

result = engine.execute(task)

print(f"\n{'=' * 70}")
print("EXECUTION SUMMARY")
print("=" * 70)
print(f"Specialists used: {result['specialists_used']}")
print(f"Quality score: {result['quality_score']}/10")
print(f"Total time: {result['total_time']}s")
print(f"\nExecution trace:")
for step in result['trace']:
    print(f"  → {step}")

print(f"\nReview:\n{result['review'][:300]}")

## 8. Visualization — ASCII Execution Graphs

Visualize the execution flow as ASCII art for debugging and documentation.

In [ ]:
class ASCIIGraphRenderer:
    """Render execution graphs as ASCII art."""

    @staticmethod
    def render_dag(nodes: List[str], edges: List[Tuple[str, str]],
                   results: Optional[Dict[str, str]] = None) -> str:
        # Build adjacency and layers
        in_deg = {n: 0 for n in nodes}
        children = defaultdict(list)
        parents = defaultdict(list)
        for src, dst in edges:
            children[src].append(dst)
            parents[dst].append(src)
            in_deg[dst] += 1

        # Assign layers via BFS
        layers = []
        remaining = set(nodes)
        while remaining:
            layer = [n for n in remaining if all(p not in remaining for p in parents[n])]
            if not layer:
                layer = [remaining.pop()]
            layers.append(sorted(layer))
            remaining -= set(layer)

        # Render
        lines = []
        lines.append("╔" + "═" * 50 + "╗")
        lines.append("║  EXECUTION GRAPH" + " " * 32 + "║")
        lines.append("╠" + "═" * 50 + "╣")

        max_width = 50
        for i, layer in enumerate(layers):
            node_strs = []
            for n in layer:
                status = "✓" if results and n in results else "○"
                node_strs.append(f"[{status} {n}]")
            layer_str = "  ".join(node_strs)
            padded = f"║  Layer {i}: {layer_str}"
            padded = padded + " " * (51 - len(padded)) + "║"
            lines.append(padded)

            # Draw edges to next layer
            if i < len(layers) - 1:
                next_layer = layers[i + 1]
                edge_parts = []
                for n in layer:
                    for c in children[n]:
                        if c in next_layer:
                            edge_parts.append(f"{n}→{c}")
                if edge_parts:
                    edge_str = ", ".join(edge_parts)
                    edge_line = f"║    ↓ {edge_str}"
                    edge_line = edge_line + " " * (51 - len(edge_line)) + "║"
                    lines.append(edge_line)

        lines.append("╚" + "═" * 50 + "╝")
        return "\n".join(lines)

# Render the DAG we executed earlier
renderer = ASCIIGraphRenderer()

nodes = ["research", "code_review", "analysis", "synthesis"]
edges = [("research", "code_review"), ("research", "analysis"),
         ("code_review", "synthesis"), ("analysis", "synthesis")]

print(renderer.render_dag(nodes, edges, {"research": "", "code_review": "", "analysis": "", "synthesis": ""}))

In [ ]:
# Render a more complex graph
complex_nodes = ["ingest", "validate", "transform_A", "transform_B",
                 "enrich", "merge", "score", "output"]
complex_edges = [
    ("ingest", "validate"),
    ("validate", "transform_A"), ("validate", "transform_B"),
    ("transform_A", "enrich"), ("transform_B", "merge"),
    ("enrich", "merge"),
    ("merge", "score"),
    ("score", "output")
]

print("Complex pipeline visualization:")
print()
print(renderer.render_dag(complex_nodes, complex_edges))
print()

# Execution stats summary
print("\n" + "=" * 50)
print("ORCHESTRATION PATTERNS SUMMARY")
print("=" * 50)
print(f"""
Pattern          │ Latency    │ Best For
─────────────────┼────────────┼─────────────────────
Router           │ +classify  │ Multi-domain dispatch
Conditional      │ Near-zero  │ Known task categories
Fan-out/Fan-in   │ N × agent  │ Multiple perspectives
DAG Executor     │ Varies     │ Complex dependencies
Graph Executor   │ Varies     │ Reusable workflows
Orchestration    │ Variable   │ Dynamic multi-step
""")

## 9. Key Takeaways

### Orchestration Patterns
1. **Router Agent** — LLM-based classification is flexible but adds latency; keyword rules are fast but rigid
2. **Conditional routing** — decision trees give predictable, fast routing; combine with LLM fallback
3. **Fan-out/Fan-in** — parallel perspectives improve breadth; aggregator synthesizes coherent output
4. **DAG execution** — topological sort ensures correct dependency order; enables complex workflows
5. **Graph executor** — reusable pattern with transforms on connections
6. **Complex orchestration** — combine triage + parallel + quality review for production-grade flows

### Design Principles
- Start simple (linear pipeline), add complexity only when needed
- Use conditional routing for known categories, LLM routing for open-ended tasks
- DAGs are the natural abstraction for multi-step workflows with dependencies
- Always include quality checks / review steps in production orchestration

### Next Steps
→ **Notebook 22**: Shared state and blackboard patterns for agent communication
→ **Notebook 23**: Swarm intelligence and emergent behavior